# Class 7: Risk

**GIS Vulnerability & Risk Assessment Course**

## Class Objectives
By the end of this lesson, you will be able to:
- Understand what **risk** means in a vulnerability assessment
- Read and apply a **risk scoring matrix** to combine probability and consequence
- Calculate risk scores for geographic features using a lookup matrix
- Visualize risk levels on a map
- Generate risk statistics and save results to a GeoPackage

## What is Risk?

**Risk** is the combination of two factors:
- **Probability**: How likely is this hazard to happen? (How often? How certain?)
- **Consequence**: How bad would it be if it happened? (What would be the impact?)

**Risk = Probability × Consequence**

A rare but catastrophic event can be a high risk. A frequent but minor event can also be a high risk. This lesson combines these two measures using a structured decision table called a **Risk Scoring Matrix**.


## Step 1: Install and Import Libraries

We will use Python libraries to work with geographic data:
- **GeoPandas**: For reading and writing geographic data (shapefiles, GeoPackages)
- **Pandas**: For working with tables of data
- **Matplotlib**: For making charts and maps
- **Folium**: For interactive maps
- **Fiona**: For reading geographic file formats

Run the cell below to install these libraries (this may take a minute).


In [ ]:
!pip install geopandas fiona shapely pyproj requests folium seaborn contextily --quiet
print("✓ Libraries installed successfully!")

Now import the libraries we just installed:


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import os
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")


## Step 2: Connect to Google Drive

Google Colab runs in the cloud. To access your files, we need to "mount" (connect to) your Google Drive. Run the cell below and follow the instructions to authorize Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")


### Optional: Run Locally Instead of Google Colab

If you prefer to run this notebook on your own machine (e.g., using Jupyter or VS Code) instead of Google Colab, comment out the Google Drive cell above and uncomment the cell below. Update the `BASE_DIR` path to point to a folder on your local filesystem.

In [ ]:
# # LOCAL RUN: Uncomment the lines below to run locally instead of Google Colab
# # Update the path to match your local folder
#
# import os
#
# BASE_DIR = '/path/to/your/local/folder/MSER_510_VULNERABILTY'
# DATA_DIR = os.path.join(BASE_DIR, 'data')
# OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
#
# os.makedirs(DATA_DIR, exist_ok=True)
# os.makedirs(OUTPUT_DIR, exist_ok=True)
#
# INPUT_GPKG = os.path.join(DATA_DIR, 'class_6_consequence.gpkg')
# OUTPUT_GPKG = os.path.join(DATA_DIR, 'class_7_risk.gpkg')
# CLASS0_GPKG = os.path.join(DATA_DIR, 'vulnerability_risk_data.gpkg')

## Step 3: Set Up Your Working Directory

All data files are stored in a folder on Google Drive. We define the path below so the code knows where to find them.


In [ ]:
BASE_DIR = '/content/drive/MyDrive/MSER_510_VULNERABILTY'
DATA_DIR = os.path.join(BASE_DIR, 'data')

# GeoPackage file paths for chained loading/saving
INPUT_GPKG = os.path.join(DATA_DIR, 'class_6_consequence.gpkg')  # Load from Class 6
OUTPUT_GPKG = os.path.join(DATA_DIR, 'class_7_risk.gpkg')  # Save to Class 7
CLASS0_GPKG = os.path.join(DATA_DIR, 'vulnerability_risk_data.gpkg')  # Fallback for base layers

# Check if the data file exists
if os.path.exists(INPUT_GPKG):
    print(f"✓ Data file found: {INPUT_GPKG}")
else:
    print(f"⚠ Warning: Data file not found at {INPUT_GPKG}")
    print(f"  Please check that you completed Class 6 first")

## Step 4: Understanding the Risk Scoring Matrix

The **Risk Scoring Matrix** is a lookup table that combines probability and consequence into a risk score. Here is how it works:

| | **Probability: Low (1)** | **Probability: Medium (2)** | **Probability: High (3)** |
|---|---|---|---|
| **Consequence: High (3)** | 2 (Medium) | 3 (High) | 3 (High) |
| **Consequence: Medium (2)** | 1 (Low) | 2 (Medium) | 3 (High) |
| **Consequence: Low (1)** | 1 (Low) | 1 (Low) | 2 (Medium) |

### How to Read the Matrix:
1. Find the row that matches your **consequence** level (top of table)
2. Find the column that matches your **probability** level (left of table)
3. The number at the intersection is your **risk score**

### Examples:
- **High consequence + High probability = High risk (3)** — This is very concerning! The hazard is likely to happen AND would cause major damage.
- **High consequence + Low probability = Medium risk (2)** — The impact would be severe, but it's unlikely to happen.
- **Low consequence + Low probability = Low risk (1)** — Minor potential impact AND it's unlikely. Not a concern.
- **Medium consequence + Medium probability = Medium risk (2)** — Moderate hazard level.

**Important Note:** This matrix is **different from the Vulnerability Matrix** in Class 4. The rows are ordered **High to Low** (from top to bottom), whereas the vulnerability matrix rows were ordered **Low to High**. Be careful when reading the correct row!


## Step 5: Visualize the Risk Scoring Matrix

The cell below creates a color-coded version of the risk matrix. This visual makes it easy to see which combinations are low, medium, or high risk at a glance:
- **Light lavender** = Low risk (1)
- **Light purple** = Medium risk (2)
- **Medium purple** = High risk (3)

Study this diagram carefully. You will use it when you calculate risk scores for your parcels!

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

# Risk matrix: rows are Consequence (High=top, Low=bottom), cols are Probability (Low, Medium, High)
# Key: (consequence, probability) → risk
risk_matrix = np.array([[3, 3, 3],   # Consequence High
                        [1, 2, 3],   # Consequence Medium
                        [1, 1, 2]])  # Consequence Low

# Color mapping: 1=light purple, 2=medium purple, 3=dark purple (official risk colors)
color_map = {1: '#D8D0E0', 2: '#B5A8C8', 3: '#7B6B95'}
text_colors = {1: 'black', 2: 'black', 3: 'white'}
risk_labels = {1: 'Low\n(1)', 2: 'Medium\n(2)', 3: 'High\n(3)'}

# Draw the matrix
for i in range(3):
    for j in range(3):
        risk_val = risk_matrix[i, j]
        color = color_map[risk_val]

        # Draw rectangle (cell)
        rect = plt.Rectangle((j, 2-i), 1, 1, linewidth=2, edgecolor='black', facecolor=color)
        ax.add_patch(rect)

        # Add text in the center of the cell
        text_color = text_colors[risk_val]
        ax.text(j+0.5, 2-i+0.5, risk_labels[risk_val],
                ha='center', va='center', fontsize=14, fontweight='bold', color=text_color)

# Set up axes
ax.set_xlim(0, 3)
ax.set_ylim(0, 3)
ax.set_aspect('equal')

# Column labels (Probability)
ax.set_xticks([0.5, 1.5, 2.5])
ax.set_xticklabels(['Low\n(1)', 'Medium\n(2)', 'High\n(3)'], fontsize=12, fontweight='bold')

# Row labels (Consequence - reversed because we draw top-to-bottom)
ax.set_yticks([0.5, 1.5, 2.5])
ax.set_yticklabels(['Low\n(1)', 'Medium\n(2)', 'High\n(3)'], fontsize=12, fontweight='bold')

# Axis labels
ax.set_xlabel('Probability →', fontsize=13, fontweight='bold')
ax.set_ylabel('Consequence ↓', fontsize=13, fontweight='bold')

# Title
ax.set_title('Risk Scoring Matrix\n(Combination of Probability & Consequence)',
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("✓ Risk matrix visualization complete!")
print("\nUse this matrix to look up risk scores:")
print("  1. Find the row for your consequence level")
print("  2. Find the column for your probability level")
print("  3. The cell at the intersection is your risk score")

## Step 6: Example Walkthrough

Let's work through some examples to practice reading the matrix:

### Example 1: Parcel in High-Risk Area
- **Consequence**: High (3) — The building is valuable and many people live there
- **Probability**: High (3) — The parcel is in the floodway; floods happen often
- **Risk**: Find row "High" and column "High" → **Risk = High (3)** ✓

**Interpretation**: This is a very serious risk. The area floods frequently AND the impact would be severe. Action needed!

### Example 2: Parcel in Unlikely but Dangerous Zone
- **Consequence**: High (3) — Large industrial facility with hazardous materials
- **Probability**: Low (1) — Earthquakes are rare in this region
- **Risk**: Find row "High" and column "Low" → **Risk = Medium (2)** ✓

**Interpretation**: Even though earthquakes are unlikely, if one occurred, the damage would be catastrophic. This still needs monitoring.

### Example 3: Low-Risk Parcel
- **Consequence**: Low (1) — Empty lot, no structures or people
- **Probability**: Medium (2) — Hazard occurs sometimes
- **Risk**: Find row "Low" and column "Medium" → **Risk = Low (1)** ✓

**Interpretation**: Even though the hazard happens occasionally, it doesn't matter much because there is nothing valuable at risk here.

### Example 4: Medium Risk
- **Consequence**: Medium (2) — Modest residential property
- **Probability**: Medium (2) — Moderate flood frequency (50-year flood zone)
- **Risk**: Find row "Medium" and column "Medium" → **Risk = Medium (2)** ✓

**Interpretation**: Moderate risk. Needs some attention but not urgent.

These examples show how **both probability AND consequence matter**. A rare but catastrophic event (Example 2) can be higher risk than a frequent but minor event.


## Step 7: Load Parcel Data from GeoPackage

Now we load your parcel data from the GeoPackage file. This file contains polygons (parcel boundaries) and columns for probability and consequence scores that were calculated in earlier classes.

The cell below:
1. Opens the GeoPackage file
2. Reads the parcel layer
3. Displays the first few rows to show what data we have


In [ ]:
# Read the parcels layer from INPUT GeoPackage (from Class 6)
parcels = gpd.read_file(INPUT_GPKG, layer='parcels')

print(f"✓ Loaded {len(parcels)} parcels from Class 6")
print(f"\nColumns in dataset:")
print(parcels.columns.tolist())
print(f"\nFirst 5 rows:")
print(parcels[['FID', 'consequence', 'probability']].head() if 'FID' in parcels.columns else parcels.head())

## Step 8: Set Up the Risk Lookup Dictionary

Now we define the risk matrix as a Python dictionary. A dictionary stores key-value pairs, making it easy to look up the risk score for any combination of probability and consequence.

The format is:
```
(consequence, probability) → risk_score
```

For example:
- `(3, 3)` means consequence=3 AND probability=3 → risk=3 (High)
- `(2, 2)` means consequence=2 AND probability=2 → risk=2 (Medium)
- `(1, 1)` means consequence=1 AND probability=1 → risk=1 (Low)

This dictionary makes the next step very simple!


In [ ]:
# Define the risk lookup matrix
# Key: (consequence, probability) → Value: risk_score
risk_matrix_dict = {
    (3, 1): 2,  # High consequence, Low probability → Medium risk
    (3, 2): 3,  # High consequence, Medium probability → High risk
    (3, 3): 3,  # High consequence, High probability → High risk
    (2, 1): 1,  # Medium consequence, Low probability → Low risk
    (2, 2): 2,  # Medium consequence, Medium probability → Medium risk
    (2, 3): 3,  # Medium consequence, High probability → High risk
    (1, 1): 1,  # Low consequence, Low probability → Low risk
    (1, 2): 1,  # Low consequence, Medium probability → Low risk
    (1, 3): 2,  # Low consequence, High probability → Medium risk
}

print("✓ Risk matrix dictionary created")
print("\nExample lookups:")
print(f"  Consequence=3, Probability=3 → Risk = {risk_matrix_dict[(3, 3)]} (High)")
print(f"  Consequence=2, Probability=2 → Risk = {risk_matrix_dict[(2, 2)]} (Medium)")
print(f"  Consequence=1, Probability=1 → Risk = {risk_matrix_dict[(1, 1)]} (Low)")


## Step 9: Calculate Risk Scores for All Parcels

Now we apply the risk matrix to each parcel. For every parcel, we look at its probability and consequence values, look them up in the dictionary, and assign the corresponding risk score.

This is done using a **lambda function** (a short, anonymous function in Python). The function looks like:
```python
lambda row: risk_matrix_dict.get((row['consequence'], row['probability']), 0)
```

In plain English: "For each row, get the consequence and probability values, look them up in the risk_matrix_dict, and return the risk score. If the combination is not found, return 0."

The `.apply()` method runs this function once for every parcel in the dataset.


In [ ]:
# Apply the risk matrix to calculate risk scores
parcels['risk'] = parcels.apply(
    lambda row: risk_matrix_dict.get((row['consequence'], row['probability']), 0),
    axis=1
)

print(f"✓ Risk scores calculated for all {len(parcels)} parcels")
print(f"\nRisk score range: {parcels['risk'].min()} to {parcels['risk'].max()}")
print(f"\nFirst 10 parcels with their probability, consequence, and calculated risk:")
print(parcels[['parno', 'probability', 'consequence', 'risk']].head(10))

## Step 9.5: Risk Results Matrix with Cross-Tabulation Statistics

The **Risk Scoring Matrix** we saw earlier (Step 4) is a **reference tool** that tells us the lookup rules. Now we will create a **Results Matrix** that shows what actually happened in our data.

The Results Matrix visualizes a **cross-tabulation** of all parcels, grouped by their probability and consequence scores. For each cell:

- **Number of parcels** in that combination
- **Total parcel value** (land value) in millions of dollars
- **Total building value** (improvement value) in millions of dollars
- **Risk score** for that combination (color-coded: light lavender=Low, light purple=Medium, medium purple=High)

This results matrix helps answer questions like:
- "How many parcels are in the high-consequence, high-probability zone?"
- "What is the total property value at risk in the worst-case scenario?"
- "Which combinations of probability and consequence actually occur in our data?"

The rows are ordered **High to Low** (same as the reference matrix in Step 5) so you can easily compare the two tables.

In [ ]:
# Helper function to format currency values
def format_currency(val):
    """Format a numeric value as a currency string"""
    if pd.isna(val) or val == 0:
        return '$0'
    if abs(val) >= 1_000_000:
        return f'${val/1_000_000:.1f}M'
    if abs(val) >= 1_000:
        return f'${val/1_000:.1f}K'
    return f'${val:.0f}'

# Define the risk matrix for looking up risk scores
risk_matrix_lookup = {
    (3, 1): 2,  # (consequence, probability) -> risk
    (3, 2): 3,
    (3, 3): 3,
    (2, 1): 1,
    (2, 2): 2,
    (2, 3): 3,
    (1, 1): 1,
    (1, 2): 1,
    (1, 3): 2,
}

# Cross-tabulation: for each (consequence, probability) pair, collect statistics
# Rows: Consequence (3=High at top, 2=Medium, 1=Low at bottom)
# Cols: Probability (1=Low, 2=Medium, 3=High left to right)
consequence_levels = [3, 2, 1]  # High to Low, top to bottom
probability_levels = [1, 2, 3]  # Low to High, left to right

# Build a matrix of results: [consequence][probability] = {count, parval, improvval}
results_matrix = {}

for cons in consequence_levels:
    for prob in probability_levels:
        # Filter parcels for this combination
        mask = (parcels['consequence'] == cons) & (parcels['probability'] == prob)
        subset = parcels[mask]
        
        # Count and sum values
        count = len(subset)
        # Handle missing fields gracefully
        parval = subset['parval'].fillna(0).sum() if 'parval' in subset.columns else 0
        improvval = subset['improvval'].fillna(0).sum() if 'improvval' in subset.columns else 0
        
        # Get risk score for this combination
        risk = risk_matrix_lookup.get((cons, prob), 0)
        
        results_matrix[(cons, prob)] = {
            'count': count,
            'parval': parval,
            'improvval': improvval,
            'risk': risk
        }

# Create the visualization
fig, ax = plt.subplots(figsize=(14, 11))

# Color mapping: risk score -> color (official risk purple palette)
color_map = {1: '#D8D0E0', 2: '#B5A8C8', 3: '#7B6B95'}
text_color_map = {1: 'black', 2: 'black', 3: 'white'}

# Draw the 3x3 grid
cell_width = 1
cell_height = 1

for i, cons in enumerate(consequence_levels):  # rows: High(3) at top
    for j, prob in enumerate(probability_levels):  # cols: Low(1) at left
        data = results_matrix[(cons, prob)]
        risk_score = data['risk']
        
        # Cell background color based on risk score
        color = color_map[risk_score]
        
        # Draw cell rectangle
        x, y = j, 2 - i  # y is inverted for top-to-bottom
        rect = plt.Rectangle((x, y), cell_width, cell_height, 
                            linewidth=2, edgecolor='black', facecolor=color)
        ax.add_patch(rect)
        
        # Determine text color based on background
        text_color = text_color_map[risk_score]
        
        # Format cell text
        cell_text = (
            f"Risk {risk_score}\n"
            f"{data['count']} parcels\n"
            f"{format_currency(data['parval'])} parcel val\n"
            f"{format_currency(data['improvval'])} bldg val"
        )
        
        # Add text to cell
        ax.text(x + cell_width/2, y + cell_height/2, cell_text,
               ha='center', va='center', fontsize=10, fontweight='bold',
               color=text_color, wrap=True)

# Set up axes
ax.set_xlim(0, 3)
ax.set_ylim(0, 3)
ax.set_aspect('equal')

# Column labels (Probability)
ax.set_xticks([0.5, 1.5, 2.5])
ax.set_xticklabels(['Prob: Low (1)', 'Prob: Medium (2)', 'Prob: High (3)'], 
                   fontsize=11, fontweight='bold')

# Row labels (Consequence, reversed because we draw top-to-bottom)
ax.set_yticks([0.5, 1.5, 2.5])
ax.set_yticklabels(['Cons: Low (1)', 'Cons: Medium (2)', 'Cons: High (3)'], 
                   fontsize=11, fontweight='bold')

# Axis labels
ax.set_xlabel('Probability →', fontsize=12, fontweight='bold')
ax.set_ylabel('Consequence ↓', fontsize=12, fontweight='bold')

# Title
ax.set_title('Risk Results Matrix: Cross-Tabulation Statistics\n(Actual Data from Parcels)',
            fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("✓ Risk Results Matrix created!")
print("\nThis matrix shows:")
print("  - Number of parcels in each (consequence, probability) combination")
print("  - Total land value (parcel value) for each combination")
print("  - Total building value (improvement value) for each combination")
print("  - Risk score (1=Low, 2=Medium, 3=High) for each combination")
print("  - Color coding matches the risk score (light purple=Low, medium purple=Medium, dark purple=High)")
print("\nCompare this Results Matrix to the Reference Matrix in Step 5.")
print("Notice which combinations have the most parcels and highest property values.")

## Step 10: Analyze Risk Statistics

Let's count how many parcels fall into each risk category and calculate percentages. This summary helps us understand the overall risk level of the study area.


In [ ]:
# Count parcels by risk level
risk_counts = parcels['risk'].value_counts().sort_index(ascending=False)
risk_labels_map = {3: 'High (3)', 2: 'Medium (2)', 1: 'Low (1)', 0: 'No Data (0)'}

print("=" * 60)
print("RISK SUMMARY STATISTICS")
print("=" * 60)
for risk_val, count in risk_counts.items():
    pct = (count / len(parcels)) * 100
    label = risk_labels_map.get(risk_val, f'Unknown ({risk_val})')
    print(f"{label:20} | Count: {count:5} | Percentage: {pct:6.2f}%")

print("-" * 60)
print(f"{'TOTAL':20} | Count: {len(parcels):5} | Percentage: 100.00%")
print("=" * 60)

# Create a DataFrame for better visualization
risk_summary = pd.DataFrame({
    'Risk Level': [risk_labels_map.get(r, f'Unknown ({r})') for r in risk_counts.index],
    'Count': risk_counts.values,
    'Percentage': (risk_counts.values / len(parcels)) * 100
})

print("\n✓ Risk summary complete")
print("\nRisk breakdown:")
print(risk_summary.to_string(index=False))


## Step 11: Visualize Risk Levels on a Map

Now we create a map showing each parcel colored by its risk level:
- **Red** = High Risk (3)
- **Yellow** = Medium Risk (2)
- **Green** = Low Risk (1)
- **Gray** = No Data (0)

This visual helps identify high-risk areas that need attention. You can see clustering of risk, which may point to neighborhoods or regions that need mitigation strategies.


In [ ]:
from IPython.display import HTML

# Define colors for risk levels - using official risk purple palette
risk_color_map = {
    3: '#7B6B95',  # Dark purple for High risk
    2: '#B5A8C8',  # Medium purple for Medium risk
    1: '#D8D0E0',  # Light purple for Low risk
    0: '#555555'   # Dark gray for No Data
}

# Create a map centered on the study area
center_lat = parcels.geometry.centroid.y.mean()
center_lon = parcels.geometry.centroid.x.mean()
risk_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
    attr='CartoDB',
    width='100%',
    height='450px'
)

# Add each parcel to the map
for idx, row in parcels.iterrows():
    risk_val = row['risk']
    color = risk_color_map.get(risk_val, '#555555')
    risk_label = risk_labels_map.get(risk_val, 'Unknown')

    # Convert geometry to GeoJSON
    if row.geometry.geom_type == 'Polygon':
        coords = list(row.geometry.exterior.coords)
        folium.Polygon(
            locations=[(lat, lon) for lon, lat in coords],
            color='#00e5ff',
            fill=True,
            fillColor=color,
            fillOpacity=0.7,
            popup=f"Parcel: {row['parno']}<br>Risk: {risk_label}",
            weight=1
        ).add_to(risk_map)
    elif row.geometry.geom_type == 'MultiPolygon':
        for part in row.geometry.geoms:
            coords = list(part.exterior.coords)
            folium.Polygon(
                locations=[(lat, lon) for lon, lat in coords],
                color='#00e5ff',
                fill=True,
                fillColor=color,
                fillOpacity=0.7,
                popup=f"Parcel: {row['parno']}<br>Risk: {risk_label}",
                weight=1
            ).add_to(risk_map)

# Add a legend
legend_html = '''
<div style="position: fixed; bottom: 50px; right: 50px; width: 200px; height: 180px; background-color: #2b2b2b; border:2px solid #00e5ff; z-index:9999; font-size:12px; padding: 10px; color: white; border-radius: 5px;">
<p style="margin: 0; font-weight: bold;">Risk Level</p>
<p style="margin: 5px 0;"><i style="background:#7B6B95; width:20px; height:20px; float:left; margin-right:8px; border: 1px solid #00e5ff;"></i>High (3)</p>
<p style="margin: 5px 0;"><i style="background:#B5A8C8; width:20px; height:20px; float:left; margin-right:8px; border: 1px solid #00e5ff;"></i>Medium (2)</p>
<p style="margin: 5px 0;"><i style="background:#D8D0E0; width:20px; height:20px; float:left; margin-right:8px; border: 1px solid #00e5ff;"></i>Low (1)</p>
<p style="margin: 5px 0;"><i style="background:#555555; width:20px; height:20px; float:left; margin-right:8px; border: 1px solid #00e5ff;"></i>No Data (0)</p>
</div>
'''
risk_map.get_root().html.add_child(folium.Element(legend_html))

risk_map.save('risk_map.html')
print("✓ Risk map created and saved as 'risk_map.html'")
print("\nMap Details:")
print(f"  Center: Latitude {center_lat:.4f}, Longitude {center_lon:.4f}")
print(f"  Total parcels displayed: {len(parcels)}")

# Display the map with HTML wrapper
map_html = risk_map._repr_html_()
display(HTML(f'<div style="height:450px;overflow:hidden;">{map_html}</div>'))

In [ ]:
import contextily as ctx
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# Reproject to Web Mercator for basemap tiles
parcels_wm = parcels.to_crs(epsg=3857)

# Create figure with map panel + legend panel below
fig = plt.figure(figsize=(10, 12))
gs = gridspec.GridSpec(2, 1, height_ratios=[10, 1.2], hspace=0.02)
ax = fig.add_subplot(gs[0])
ax_legend = fig.add_subplot(gs[1])

# Layer 1 (bottom): All parcels - no fill, thin grey borders
parcels_wm.plot(ax=ax, facecolor='none', edgecolor='#888888', linewidth=0.3)

# Layer 2: Risk layer - score 0 NOT plotted
risk_colors = {1: '#D8D0E0', 2: '#B5A8C8', 3: '#7B6B95'}
for score in [3, 2, 1]:  # Plot high scores first so low scores render on top
    subset = parcels_wm[parcels_wm['risk'] == score]
    if len(subset) > 0:
        subset.plot(ax=ax, facecolor=risk_colors[score], edgecolor='none', alpha=0.85)

# Layer 3: Buildings
try:
    buildings_layer = gpd.read_file(CLASS0_GPKG, layer='buildings')
    buildings_wm = buildings_layer.to_crs(epsg=3857)
    buildings_wm.plot(ax=ax, facecolor='#3D3D3D', edgecolor='#2a2a2a', linewidth=0.1, alpha=0.7)
    print(f"✓ Buildings loaded: {len(buildings_layer)} features")
except Exception as e:
    print(f"⚠ Could not load buildings: {e}")

# Layer 4: Flood zones with transparency (all three categories)
try:
    flood_zones_full = gpd.read_file(CLASS0_GPKG, layer='flood_zones')
    flood_wm = flood_zones_full.to_crs(epsg=3857)
    flood_colors = {'Floodway': '#2B5797', '100-year': '#8FABBE', '500-year': '#B4D4E7'}
    for flood_type in ['500-year', '100-year', 'Floodway']:
        flood_subset = flood_wm[flood_wm['flood_category'] == flood_type]
        if len(flood_subset) > 0:
            flood_subset.plot(ax=ax, facecolor=flood_colors.get(flood_type, '#B4D4E7'),
                            edgecolor='none', alpha=0.5)
    print(f"✓ Flood zones loaded: {len(flood_zones_full)} features")
    print(f"  Categories: {flood_zones_full['flood_category'].value_counts().to_dict()}")
except Exception as e:
    print(f"⚠ Could not load flood zones: {e}")

# Add CartoDB Positron (light) basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom='auto')

# Style the map panel
ax.set_axis_off()
ax.set_title('Risk Assessment', fontsize=14, fontweight='bold', pad=10)

# Build legend in bottom panel
ax_legend.set_axis_off()
legend_elements = [
    Patch(facecolor='#7B6B95', edgecolor='none', label='High (3)'),
    Patch(facecolor='#B5A8C8', edgecolor='none', label='Medium (2)'),
    Patch(facecolor='#D8D0E0', edgecolor='none', label='Low (1)'),
    Patch(facecolor='none', edgecolor='none', label=''),  # spacer
    Patch(facecolor='#3D3D3D', edgecolor='#2a2a2a', label='Buildings'),
    Patch(facecolor='#2B5797', edgecolor='none', alpha=0.5, label='Floodway'),
    Patch(facecolor='#8FABBE', edgecolor='none', alpha=0.5, label='100-year Floodplain'),
    Patch(facecolor='#B4D4E7', edgecolor='none', alpha=0.5, label='500-year Floodplain'),
    Patch(facecolor='none', edgecolor='#888888', linewidth=0.5, label='Parcels'),
]
ax_legend.legend(handles=legend_elements, loc='center', ncol=4, fontsize=9,
                frameon=True, facecolor='white', edgecolor='#cccccc',
                handlelength=1.5, handletextpad=0.5, columnspacing=1.5)

# Export
# Ensure OUTPUT_DIR is defined and created
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
png_path = os.path.join(OUTPUT_DIR, 'risk_map.png')
plt.savefig(png_path, dpi=150, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f"✓ Map exported to: {png_path}")

## Step 12.5: Export Risk Map as PNG

Now we'll create a publication-quality PNG map that combines all layers in the proper order with dark styling. This map shows risk assessment with all supporting layers (buildings, flood zones, and parcels) in a single exportable image.

## Step 12: Create a Chart of Risk Distribution

Let's visualize the risk distribution as a bar chart. This makes it easy to see at a glance how many parcels are in each risk category.


In [ ]:
# Create a bar chart of risk distribution
fig, ax = plt.subplots(figsize=(10, 6))

# Data for the chart (sorted from High to Low risk)
risk_order = [3, 2, 1, 0]
risk_labels_chart = ['High (3)', 'Medium (2)', 'Low (1)', 'No Data (0)']
risk_colors_chart = ['#7B6B95', '#B5A8C8', '#D8D0E0', '#CCCCCC']

counts = [len(parcels[parcels['risk'] == r]) for r in risk_order]

# Create bar chart
bars = ax.bar(risk_labels_chart, counts, color=risk_colors_chart, edgecolor='black', linewidth=1.5)

# Add count labels on top of bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_xlabel('Risk Level', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Parcels', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Parcels by Risk Level', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Risk distribution chart created")

## Step 13: Save Risk Scores to GeoPackage

Now we save the parcels with their calculated risk scores back to the GeoPackage file. This preserves all the geographic information and allows other tools (like QGIS or ArcGIS) to access the risk data later.

We save it as a new layer called 'parcels_with_risk' so we don't overwrite the original data.


In [ ]:
try:
    import fiona
    import sqlite3
    
    # Save parcels layer first (creates new GeoPackage)
    parcels.to_file(OUTPUT_GPKG, layer='parcels', driver='GPKG', mode='w')
    print(f"✓ Saved parcels layer with risk ({len(parcels)} features)")
    
    # Copy forward all other layers from the input GeoPackage
    if os.path.exists(INPUT_GPKG):
        input_layers = fiona.listlayers(INPUT_GPKG)
        for layer_name in input_layers:
            if layer_name == 'parcels':
                continue  # Already saved updated version
            try:
                layer_data = gpd.read_file(INPUT_GPKG, layer=layer_name)
                layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                print(f"✓ Copied layer: {layer_name} ({len(layer_data)} features)")
            except Exception as e:
                print(f"  Note: Could not copy layer '{layer_name}' as spatial: {e}")
        
        # Also copy any non-spatial tables (like 'summary') via sqlite3
        try:
            conn_in = sqlite3.connect(INPUT_GPKG)
            conn_out = sqlite3.connect(OUTPUT_GPKG)
            cursor = conn_in.cursor()
            # Get all tables that aren't in fiona's layer list and aren't system tables
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
            all_tables = [row[0] for row in cursor.fetchall()]
            system_tables = ['gpkg_contents', 'gpkg_geometry_columns', 'gpkg_spatial_ref_sys',
                            'gpkg_ogr_contents', 'gpkg_tile_matrix', 'gpkg_tile_matrix_set',
                            'sqlite_sequence', 'gpkg_extensions', 'gpkg_metadata',
                            'gpkg_metadata_reference']
            for table in all_tables:
                if table in system_tables or table in input_layers or table.startswith('rtree_') or table.startswith('trigger_'):
                    continue
                try:
                    df = pd.read_sql(f'SELECT * FROM "{table}"', conn_in)
                    if len(df) > 0:
                        df.to_sql(table, conn_out, if_exists='replace', index=False)
                        print(f"✓ Copied non-spatial table: {table} ({len(df)} rows)")
                except Exception:
                    pass
            conn_in.close()
            conn_out.close()
        except Exception:
            pass
    
    # Also ensure base layers from Class 0 are included (flood_zones, buildings, study_area)
    # These may not be in INPUT_GPKG if earlier classes didn't carry them forward
    if os.path.exists(CLASS0_GPKG):
        try:
            import fiona as _fiona
            # Get layers already written to output
            output_layers = _fiona.listlayers(OUTPUT_GPKG)
            # Get layers available in Class 0
            class0_layers = _fiona.listlayers(CLASS0_GPKG)
            # Copy any missing layers
            for layer_name in class0_layers:
                if layer_name not in output_layers:
                    try:
                        layer_data = gpd.read_file(CLASS0_GPKG, layer=layer_name)
                        layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                        print(f"✓ Added base layer from Class 0: {layer_name} ({len(layer_data)} features)")
                    except Exception as e:
                        print(f"  Note: Could not copy base layer '{layer_name}': {e}")
        except Exception:
            pass
    
    print(f"\n✓ All data saved to: {OUTPUT_GPKG}")
except Exception as e:
    print(f"✗ Error saving to GeoPackage: {e}")
    print("\nAlternative: Save to a new GeoPackage file")
    alt_path = os.path.join(DATA_DIR, 'class_7_risk_backup.gpkg')
    parcels.to_file(alt_path, layer='parcels', driver='GPKG')
    print(f"✓ Saved to: {alt_path}")

## Step 14: Summary and Key Takeaways

Congratulations! You have successfully calculated risk scores for all parcels in your study area. Here is what you learned:

### Key Concepts:
1. **Risk = Probability × Consequence** — Risk combines how likely something is with how bad it would be
2. **Risk Scoring Matrix** — A structured lookup table that combines probability and consequence into a single risk score
3. **Risk Levels**:
   - **High (3)**: Urgent attention needed. High likelihood AND high impact.
   - **Medium (2)**: Moderate concern. Needs monitoring and mitigation planning.
   - **Low (1)**: Minor risk. Can often be accepted or mitigated with simple measures.

### What You Did:
1. ✓ Loaded parcel data with probability and consequence scores
2. ✓ Applied the risk matrix to calculate a risk score for each parcel
3. ✓ Analyzed risk statistics (counts and percentages)
4. ✓ Created a risk map showing spatial distribution of risk
5. ✓ Saved results to a GeoPackage for future use

### Next Steps:
- Use the risk map to identify priority areas for mitigation
- Consider adaptation and mitigation strategies for high-risk parcels
- Compare risk levels across different neighborhoods or zones
- Share results with stakeholders and planners

### Important Reminders:
- Always check the **row order** when reading a matrix. This risk matrix has rows ordered High→Low (different from the vulnerability matrix in Class 4 which was Low→High)
- Risk is not the same as **vulnerability** or **hazard exposure**. It combines both likelihood and consequence.
- Risk assessments are tools to inform decision-making, not absolute predictions of the future.


## Color Reference for GIS Symbology

Use these hex color values when styling your risk layer in QGIS or ArcGIS Pro:

| Score | Label | Hex Code | RGB |
|-------|-------|----------|-----|
| 1 | Low | `#D8D0E0` | 216, 208, 224 |
| 2 | Medium | `#B5A8C8` | 181, 168, 200 |
| 3 | High | `#7B6B95` | 123, 107, 149 |

**How to apply in QGIS:**
1. Right-click your layer → Properties → Symbology
2. Choose "Categorized" from the dropdown
3. Set Column to `risk`
4. Click "Classify"
5. Double-click each symbol to change its color using the hex values above

**How to apply in ArcGIS Pro:**
1. Right-click your layer → Symbology
2. Choose "Unique Values"
3. Set Field 1 to `risk`
4. Click "Add all values"
5. Double-click each symbol to change its color using the hex values above

**Pre-made symbology files** are also available in the `symbology/` folder:
- `risk_symbology.qml` — Load in QGIS via Style → Load Style
- `risk_symbology.lyrx` — Import in ArcGIS Pro via Symbology → Import


## Appendix A: How to Do This in QGIS

If you want to calculate risk scores in QGIS instead of Python, use a **Field Calculator** with this expression:

```
CASE
  WHEN "consequence" = 3 AND "probability" = 1 THEN 2
  WHEN "consequence" = 3 AND "probability" = 2 THEN 3
  WHEN "consequence" = 3 AND "probability" = 3 THEN 3
  WHEN "consequence" = 2 AND "probability" = 1 THEN 1
  WHEN "consequence" = 2 AND "probability" = 2 THEN 2
  WHEN "consequence" = 2 AND "probability" = 3 THEN 3
  WHEN "consequence" = 1 AND "probability" = 1 THEN 1
  WHEN "consequence" = 1 AND "probability" = 2 THEN 1
  WHEN "consequence" = 1 AND "probability" = 3 THEN 2
  ELSE 0
END
```

### Steps in QGIS:
1. Load your parcel layer in QGIS
2. Open the attribute table (right-click layer → Open Attribute Table)
3. Click **Field Calculator** (function icon in toolbar)
4. Create a new field called `risk` (type: Integer)
5. Paste the CASE expression above
6. Click OK to calculate for all features
7. Save the layer

The result will be identical to what we calculated in Python.


## Appendix B: How to Do This in ArcGIS Pro

In ArcGIS Pro, you can create a custom function in Python and use it in the **Field Calculator**.

### Step 1: Create a Python Function
```python
def calc_risk(consequence, probability):
    """Calculate risk from consequence and probability"""
    matrix = {
        (3, 1): 2, (3, 2): 3, (3, 3): 3,
        (2, 1): 1, (2, 2): 2, (2, 3): 3,
        (1, 1): 1, (1, 2): 1, (1, 3): 2,
    }
    return matrix.get((consequence, probability), 0)
```

### Step 2: Use in Field Calculator
In the Field Calculator expression box, enter:
```
calc_risk(!consequence!, !probability!)
```

### Step 3: Set Result Type
- Result Type: **Short Integer** or **Integer**
- Field Name: `risk`

### Step 4: Execute
Click **OK** to calculate the risk field for all features.

The result will match our Python and QGIS calculations exactly.

### Note:
If you want to use this in ArcGIS ModelBuilder or other workflows, you can save the function in a `.py` file in your Python Toolbox and call it from there.


## Appendix C: Troubleshooting

### Problem: "GeoPackage file not found"
**Solution**:
- Check that the file `vulnerability_risk_data.gpkg` exists in your Google Drive folder `MSER_510_VULNERABILTY`
- Make sure you authorized Google Colab to access your Drive (Step 2)
- If needed, upload the file to the correct folder

### Problem: "No such layer: parcels"
**Solution**:
- The GeoPackage might have a different layer name
- Run this code to see what layers are available:
  ```python
  import fiona
  layers = fiona.listlayers(gpkg_path)
  print(layers)
  ```
- Change `'parcels'` to the correct layer name

### Problem: "KeyError" when looking up risk
**Solution**:
- Some parcels might have missing probability or consequence values (NaN)
- The code handles this by returning 0, which we label as "No Data"
- Check the data quality: `print(parcels[['probability', 'consequence']].describe())`

### Problem: Map doesn't display correctly
**Solution**:
- The map is saved to `risk_map.html` in your Colab workspace
- If geometries are not valid, try: `parcels = parcels[parcels.geometry.is_valid]`
- Make sure geometries are in WGS84 (EPSG:4326) for Folium: `parcels = parcels.to_crs('EPSG:4326')`
